In [9]:
from dotenv import load_dotenv
load_dotenv()
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [10]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
def file_reader(directory_path):
    """Reads all PDF files in a directory."""
    
    # ... (Your loading logic) ...
    loader = DirectoryLoader(directory_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    
    # --- THE FIX: Convert Objects to Dictionaries ---
    clean_data = []
    for doc in documents:
        clean_data.append({
            "page_content": doc.page_content,
        })
    return clean_data
data=file_reader("/Users/oluwaferanmioyelude/Documents/Syllabus")
for i in data:
    print(i)# Print the first 500 characters of each document

{'page_content': 'ENGW  104–  Writing,  Literacy,  and  Discourse  \nSpring  2026  \nCourse  Instructor:  Prof.  Kenyatta  Graves  \nClass  Time:  Section  27—MWF  10:10a-11:00a;  Section  28—MWF  11:10a-12:00p  \nEmail  Address:  kenyatta.graves@howard.edu  \nOffice  Hours:  Mondays  and  Wednesday—2:00pm  to  5:00pm;  Tuesdays  4:00pm-7:00pm.  \nAll\n \noffice\n \nhours\n \nare\n \nvirtual,\n \nvia\n \nZoom\n \nand\n \nby\n \nappointment\n \nonly.\n \nAdditional\n \ndays/hours\n \navailable\n \nby\n \nrequest\n \nvia\n \nemail.\n \nWriting  For  Your  Life:  Identity,  Space,  and  Place  \n“Word-work  is  sublime,  she  thinks,  because  it  is  generative;  it  makes  meaning  that  secures  \nour\n \ndifference,\n \nour\n \nhuman\n \ndifference\n \n–\n \nthe\n \nway\n \nin\n \nwhich\n \nwe\n \nare\n \nlike\n \nno\n \nother\n \nlife.\n \nWe\n \ndie.\n \nThat\n \nmay\n \nbe\n \nthe\n \nmeaning\n \nof\n \nlife.\n \nBut\n \nwe\n \ndo\n \nlanguage.\n \nThat\n \nmay\n \nbe\n \nthe\n \nm

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,)


In [12]:
from system_prompt import SYSTEM_PROMPT

In [13]:
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
class Strategy(BaseModel):
    action: str = Field(description="Action to take, e.g. 'Start Studying'")
    start_date: str = Field(description="ISO date YYYY-MM-DD")
    reasoning: str = Field(description="Why this date?")

class Event(BaseModel):
    title: str
    type: str
    due_date: str
    priority: str
    strategy: Optional[Strategy] = None

class SyllabusData(BaseModel):
    course_name: str
    events: List[Event]
# 3. The Single Course (Renamed for clarity)
class CourseSyllabus(BaseModel):
    course_name: str
    events: List[Event]

# 4. THE NEW MASTER CONTAINER
class SemesterPlan(BaseModel):
    courses: List[CourseSyllabus]

In [14]:
from langchain_core.prompts import ChatPromptTemplate
def content_structurer(file_content: List[Dict]):
    combined_text = "\n".join([page['page_content'] for page in file_content])
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("user", "Here is the syllabus text: {raw_text}")
        ])
    structured_llm = llm.with_structured_output(SemesterPlan, method="json_mode")
    chain=  prompt | structured_llm
    result = chain.invoke({"raw_text": combined_text})
    return result.model_dump_json()
structured_data = content_structurer(data)

In [15]:
structured_data

'{"courses":[{"course_name":"ENGW 104","events":[{"title":"Personal Declaration Essay","type":"assignment","due_date":"2026-01-21","priority":"high","strategy":{"action":"Start Working","start_date":"2026-01-16","reasoning":"2 days lead time for assignments"}},{"title":"Personal Declaration Essay","type":"assignment","due_date":"2026-01-30","priority":"high","strategy":{"action":"Start Working","start_date":"2026-01-25","reasoning":"2 days lead time for assignments"}},{"title":"Personal Declaration Essay","type":"assignment","due_date":"2026-02-11","priority":"high","strategy":{"action":"Start Working","start_date":"2026-02-06","reasoning":"2 days lead time for assignments"}},{"title":"Personal Declaration Essay","type":"assignment","due_date":"2026-02-13","priority":"high","strategy":{"action":"Start Working","start_date":"2026-02-08","reasoning":"2 days lead time for assignments"}},{"title":"Identity and Place Essay","type":"assignment","due_date":"2026-02-16","priority":"high","stra